In [1]:
import json
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

# D13에서 확인한 MSFT의 기업번호
cik = "0000789019"

# D14는 새 노트북이므로 연락처를 다시 입력
email = input("연락 이메일만 입력하고 Enter: ").strip()

if not email or "@" not in email or any(c.isspace() for c in email):
    raise ValueError("이메일 주소만 입력하세요.")

url = f"https://data.sec.gov/submissions/CIK{cik}.json"

request = Request(
    url,
    headers={
        "User-Agent": f"Twodigit Research {email}",
        "Accept": "application/json",
    },
)

try:
    with urlopen(request, timeout=30) as response:
        http_status = response.status
        submissions = json.loads(response.read().decode("utf-8"))

    print("현재 폴더:", Path.cwd())
    print("HTTP 상태:", http_status)
    print("회사명:", submissions["name"])
    print("CIK:", str(submissions["cik"]).zfill(10))

    recent = submissions["filings"]["recent"]
    print("최근 공시 기록 수:", len(recent["accessionNumber"]))
    print("추가 과거 이력 파일 수:", len(submissions["filings"]["files"]))

except HTTPError as error:
    print("HTTP 오류:", error.code)
except URLError as error:
    print("네트워크 오류:", error.reason)

연락 이메일만 입력하고 Enter:  skkufi0113@skkukdp.re.kr


현재 폴더: C:\ai_campus_main_proj\data_research\D14
HTTP 상태: 200
회사명: MICROSOFT CORP
CIK: 0000789019
최근 공시 기록 수: 1002
추가 과거 이력 파일 수: 2


In [2]:
# SEC 응답은 항목별 리스트이므로, 공시 한 건씩 묶기
fields = [
    "form",
    "filingDate",
    "reportDate",
    "accessionNumber",
    "primaryDocument",
]

lengths = {field: len(recent[field]) for field in fields}
print("항목별 기록 수:", lengths)

if len(set(lengths.values())) != 1:
    raise ValueError("항목별 길이가 달라 공시를 정확히 묶을 수 없습니다.")

filings = [
    dict(zip(fields, values))
    for values in zip(*(recent[field] for field in fields))
]

# 주요 보고서와 정정 보고서 선택
target_forms = {"10-K", "10-Q", "8-K", "10-K/A", "10-Q/A", "8-K/A"}

selected = sorted(
    [row for row in filings if row["form"] in target_forms],
    key=lambda row: (row["filingDate"], row["accessionNumber"]),
    reverse=True,
)

print("\n주요 공시 최근 10건")
for row in selected[:10]:
    print(
        f"{row['form']} | 제출일: {row['filingDate']} | "
        f"보고기준일: {row['reportDate']} | "
        f"접수번호: {row['accessionNumber']} | "
        f"문서: {row['primaryDocument']}"
    )

print("\n추가 과거 이력 파일")
for item in submissions["filings"]["files"]:
    print(
        f"파일: {item['name']} | "
        f"기간: {item.get('filingFrom')} ~ {item.get('filingTo')} | "
        f"건수: {item.get('filingCount')}"
    )

항목별 기록 수: {'form': 1002, 'filingDate': 1002, 'reportDate': 1002, 'accessionNumber': 1002, 'primaryDocument': 1002}

주요 공시 최근 10건
8-K | 제출일: 2026-09-02 | 보고기준일: 2026-09-02 | 접수번호: 0001193125-26-380280 | 문서: d291965d8k.htm
10-K | 제출일: 2026-07-29 | 보고기준일: 2026-06-30 | 접수번호: 0001193125-26-323660 | 문서: msft-20260630.htm
8-K | 제출일: 2026-07-29 | 보고기준일: 2026-07-29 | 접수번호: 0001193125-26-323632 | 문서: msft-20260729.htm
8-K | 제출일: 2026-06-05 | 보고기준일: 2026-06-02 | 접수번호: 0001193125-26-258667 | 문서: d26760d8k.htm
8-K | 제출일: 2026-05-14 | 보고기준일: 2026-05-13 | 접수번호: 0001193125-26-224155 | 문서: d125909d8k.htm
10-Q | 제출일: 2026-04-29 | 보고기준일: 2026-03-31 | 접수번호: 0001193125-26-191507 | 문서: msft-20260331.htm
8-K | 제출일: 2026-04-29 | 보고기준일: 2026-04-29 | 접수번호: 0001193125-26-191457 | 문서: msft-20260429.htm
10-Q | 제출일: 2026-01-28 | 보고기준일: 2025-12-31 | 접수번호: 0001193125-26-027207 | 문서: msft-20251231.htm
8-K | 제출일: 2026-01-28 | 보고기준일: 2026-01-28 | 접수번호: 0001193125-26-027198 | 문서: msft-20260128.htm
8-K | 제출일: 2025-12-08 |

In [3]:
# 1. 과거 이력 파일 1개 조회
history_info = submissions["filings"]["files"][0]
history_url = (
    "https://data.sec.gov/submissions/" + history_info["name"]
)

history_request = Request(
    history_url,
    headers={
        "User-Agent": f"Twodigit Research {email}",
        "Accept": "application/json",
    },
)

with urlopen(history_request, timeout=30) as response:
    history_status = response.status
    history = json.loads(response.read().decode("utf-8"))

print("과거 이력 HTTP 상태:", history_status)
print("과거 이력 실제 건수:", len(history["accessionNumber"]))
print(
    "실제 제출일 범위:",
    min(history["filingDate"]),
    "~",
    max(history["filingDate"]),
)

# 2. 최근 10-K 선택
annual_reports = [row for row in selected if row["form"] == "10-K"]

if not annual_reports:
    raise ValueError("최근 목록에서 10-K를 찾지 못했습니다.")

report = annual_reports[0]

# 원문 주소에서는 CIK의 앞자리 0과 접수번호의 하이픈 제거
document_url = (
    f"https://www.sec.gov/Archives/edgar/data/"
    f"{int(cik)}/"
    f"{report['accessionNumber'].replace('-', '')}/"
    f"{report['primaryDocument']}"
)

print("\nD15에서 확인할 공시")
print("종류:", report["form"])
print("제출일:", report["filingDate"])
print("보고기준일:", report["reportDate"])
print("원문 URL:", document_url)

과거 이력 HTTP 상태: 200
과거 이력 실제 건수: 2002
실제 제출일 범위: 2008-08-13 ~ 2020-08-05

D15에서 확인할 공시
종류: 10-K
제출일: 2026-07-29
보고기준일: 2026-06-30
원문 URL: https://www.sec.gov/Archives/edgar/data/789019/000119312526323660/msft-20260630.htm


In [5]:
import csv
from collections import Counter
from pathlib import Path

# 1. 최근 목록의 결측 확인
print("[최근 공시 결측]")
for field in fields:
    count = sum(
        row.get(field) is None or str(row.get(field)).strip() == ""
        for row in filings
    )
    print(f"{field}: {count}건")

# 2. 공시 식별번호 중복 확인
counts = Counter(row["accessionNumber"] for row in filings)
print("\n접수번호 중복 추가 건수:",
      sum(count - 1 for count in counts.values()))

# 3. 실제 제공 기간과 공시 종류 확인
dates = [row["filingDate"] for row in filings if row["filingDate"]]
print("최근 목록 실제 제출일 범위:", min(dates), "~", max(dates))

for form in ["10-K", "10-Q", "8-K"]:
    print(f"{form}: {sum(row['form'] == form for row in filings)}건")

print("정정 공시:",
      sum(str(row["form"]).endswith("/A") for row in filings), "건")

# 4. 주요 공시 최근 10건에 원문 URL 추가
sample_rows = []

for row in selected[:10]:
    item = row.copy()
    item["document_url"] = (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{int(cik)}/"
        f"{row['accessionNumber'].replace('-', '')}/"
        f"{row['primaryDocument']}"
    )
    sample_rows.append(item)

if not sample_rows:
    raise ValueError("저장할 공시가 없습니다.")

sample_path = Path.cwd() / "D14_MSFT_filings_sample.csv"

with sample_path.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=list(sample_rows[0]))
    writer.writeheader()
    writer.writerows(sample_rows)

# 5. 저장 결과 재확인
with sample_path.open(newline="", encoding="utf-8-sig") as file:
    saved_rows = list(csv.DictReader(file))

print("\n저장 위치:", sample_path)
print("저장 공시 수:", len(saved_rows))
print("CSV 내용 일치:", saved_rows == sample_rows)

[최근 공시 결측]
form: 0건
filingDate: 0건
reportDate: 133건
accessionNumber: 0건
primaryDocument: 0건

접수번호 중복 추가 건수: 0
최근 목록 실제 제출일 범위: 2020-08-07 ~ 2026-09-17
10-K: 6건
10-Q: 18건
8-K: 56건
정정 공시: 16 건

저장 위치: C:\ai_campus_main_proj\data_research\D14\D14_MSFT_filings_sample.csv
저장 공시 수: 10
CSV 내용 일치: True
